# Modelo jerárquico para recepciones y yardas

## Qué responde este notebook

[`4.7_jerarquico_touchdowns.ipynb`](4.7_jerarquico_touchdowns.ipynb) probó un modelo jerárquico
(un efecto por jugador, encogido hacia el promedio de la posición) para touchdowns. Aquí se prueba
el mismo enfoque, con las mismas 24 variables de la versión lineal, en `receptions` y
`receiving_yards`, contra el XGBoost elegido en
[`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb), con la regla de
[`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb): RMSE en validación
(2022-2023) y adopción solo si la mejora tiene un intervalo bootstrap que no incluye cero. La prueba
(2024-2025) solo se reporta. Las variables están descritas en el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md).

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import time
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
features_ridge = features.columnas_por_modelo("ridge")
features_arbol = features.columnas_por_modelo("arbol")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(wr, range(2016, 2022), [2022, 2023], [2024, 2025])
cols_numericas = [c for c in features_ridge if c != "anios_experiencia_bucket"]

CONFIG_ELEGIDA = {
    "receptions": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03),
    "receiving_tds": dict(objective="count:poisson", max_depth=3, n_estimators=400, learning_rate=0.03),
}

def imputar(df, ajustar, imp_num, imp_cat):
    fn_num = imp_num.fit_transform if ajustar else imp_num.transform
    fn_cat = imp_cat.fit_transform if ajustar else imp_cat.transform
    num = pd.DataFrame(fn_num(df[cols_numericas]), index=df.index, columns=cols_numericas)
    cat = pd.DataFrame(fn_cat(df[["anios_experiencia_bucket"]]), index=df.index, columns=["anios_experiencia_bucket"])
    return pd.concat([num, cat], axis=1)[features_ridge]

def datos_bambi(mascara, imp_num, imp_cat, ajustar=False):
    datos = imputar(wr.loc[mascara, features_ridge], ajustar, imp_num, imp_cat)
    datos["player_id"] = wr.loc[mascara, "player_id"].astype(str).to_numpy()
    return datos

def predecir_mu(modelo, idata, datos):
    p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)
    return p.posterior["mu"].mean(dim=["chain", "draw"]).to_numpy()

def xgb_elegido(target):
    modelo = XGBRegressor(**CONFIG_ELEGIDA[target], random_state=0, n_jobs=-1)
    modelo.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
    return pd.Series(np.clip(modelo.predict(wr[features_arbol]), 0, None), index=wr.index)

def comparar(target, predicciones):
    """Metricas de validacion y prueba de cada prediccion, y diferencia de RMSE de validacion
    contra el XGBoost elegido con su intervalo bootstrap por semanas."""
    y = wr[target]
    media = y[train_mask].mean()
    filas = {}
    for nombre, pred in predicciones.items():
        fila = {}
        for parte, mascara in (("val", val_mask), ("test", test_mask)):
            m = modeling.metricas_target(target, y[mascara], pred[mascara], media)
            fila.update({f"{parte}_{k}": m[k] for k in ("rmse", "r2_oos", "sesgo", "pendiente_calibracion", "mae")})
        if nombre != "XGBoost elegido":
            c = experimentos.diferencia_bootstrap(wr[val_mask], y[val_mask], pred[val_mask],
                                                  predicciones["XGBoost elegido"][val_mask], "rmse")
            fila.update({"dif_val_rmse": c["diferencia"], "IC_inf": c["ic_inferior"], "IC_sup": c["ic_superior"],
                         "se_adopta": (not c["empate"]) and c["diferencia"] < 0})
        filas[nombre] = fila
    return pd.DataFrame(filas).T.infer_objects()

## 1. Recepciones

### Qué familia usar

Las recepciones son un conteo entero. La Poisson supone varianza igual a la media; si la varianza es
mayor (sobredispersión), la Binomial Negativa es la elección estándar.

In [2]:
y_rec = wr.loc[train_mask, "receptions"]
print(f"media {y_rec.mean():.3f}, varianza {y_rec.var():.3f}, varianza / media = {y_rec.var() / y_rec.mean():.2f}")
print(f"semanas con 0 recepciones: {(y_rec == 0).mean():.1%}")

media 2.765, varianza 6.493, varianza / media = 2.35
semanas con 0 recepciones: 21.2%


**Lectura.** En entrenamiento la varianza de las recepciones es 2.35 veces su media (6.493 contra
2.765) y 21.2% de las semanas terminan en cero. Hay sobredispersión clara, así que se usa la
Binomial Negativa, la misma familia que en touchdowns. Parte de esa varianza de más viene de las
diferencias entre receptores, que el modelo capta con sus variables y el efecto por jugador; la
Binomial Negativa estima aparte cuánta dispersión queda, y si queda poca se acerca a una Poisson.

In [3]:
target = "receptions"
imp_num_rec, imp_cat_rec = SimpleImputer(strategy="median"), SimpleImputer(strategy="most_frequent")
datos_train_rec = datos_bambi(train_mask, imp_num_rec, imp_cat_rec, ajustar=True)
datos_train_rec[target] = wr.loc[train_mask, target].to_numpy()

formula_rec = f"{target} ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo_rec = bmb.Model(formula_rec, datos_train_rec, family="negativebinomial")
t0 = time.time()
idata_rec = modelo_rec.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste: {time.time() - t0:.0f}s")

resumen_rec = az.summary(idata_rec)
print(f"r_hat maximo: {resumen_rec['r_hat'].max():.4f} | parametros con r_hat > 1.01: "
      f"{(resumen_rec['r_hat'] > 1.01).sum()} de {len(resumen_rec)}")

Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, Intercept, receptions_career_avg, depth_team, receptions_season_avg, receptions_last3_avg, fantasy_points_ppr_last5_avg, total_implicito_equipo, receiving_yards_after_catch_career_avg, receiving_yards_career_avg, wopr_last3_avg, receiving_yards_last5_avg, fantasy_points_ppr_career_avg, receptions_last5_avg, fantasy_points_ppr_last3_avg, wopr_season_avg, target_share_x_ofensiva_equipo, targets_season_avg, draft_pick, receiving_first_downs_career_avg, receiving_yards_after_catch_season_avg, targets_last5_avg, edad, anios_experiencia, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


/usr/local/lib/python3.13/site-packages/pymc/step_methods/hmc/quadpotential.py:321: RuntimeWarning: overflow encountered in dot
  return 0.5 * np.dot(x, v_out)


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 314 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste: 327s


r_hat maximo: 1.0357 | parametros con r_hat > 1.01: 44 de 534


In [4]:
pred_jerarquico_rec = pd.Series(np.nan, index=wr.index)
for mascara in (val_mask, test_mask):
    pred_jerarquico_rec[mascara] = predecir_mu(modelo_rec, idata_rec, datos_bambi(mascara, imp_num_rec, imp_cat_rec))

comparacion_rec = comparar("receptions", {
    "XGBoost elegido": xgb_elegido("receptions"),
    "Jerárquico + Binomial Negativa": pred_jerarquico_rec,
})
comparacion_rec.T

/tmp/ipykernel_7/2073560033.py:40: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


/tmp/ipykernel_7/2073560033.py:40: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


,XGBoost elegido,Jerárquico + Binomial Negativa
val_rmse,1.931488,2.038914
val_r2_oos,0.436423,0.371989
val_sesgo,0.091884,0.111956
val_pendiente_calibracion,1.047423,0.944896
val_mae,1.450159,1.547247
test_rmse,1.835523,1.978066
test_r2_oos,0.452724,0.364423
test_sesgo,0.057671,-0.087512
test_pendiente_calibracion,1.021196,0.947152
test_mae,1.369657,1.474716


**Lectura.** El jerárquico queda claramente por detrás. En validación su RMSE es 2.039 contra 1.931 del
XGBoost elegido: una diferencia de +0.107 con intervalo de +0.085 a +0.132, peor de forma real. Su R²
fuera de muestra baja de 0.436 a 0.372, y en prueba el orden se mantiene (1.978 contra 1.836). La
convergencia también es más débil que en touchdowns: 44 de 534 parámetros con `r_hat` sobre 1.01
(máximo 1.0357) y avisos de tamaño de muestra efectivo bajo. Eso resta confianza a la estimación de
cada parámetro, pero es poco probable que explique una diferencia de ese tamaño; aquí no se prueba
con cadenas más largas.

## 2. Yardas

### Escala y valores negativos

Las yardas toman muchos valores y tienen cola larga a la derecha (ver
[`2.1_eda_targets.ipynb`](2.1_eda_targets.ipynb)). Una opción común es modelarlas en escala
logarítmica con una familia Gaussiana. Antes conviene revisar si son siempre positivas.

In [5]:
negativos = wr.loc[train_mask & (wr["receiving_yards"] < 0)]
print(f"filas de entrenamiento con receiving_yards < 0: {len(negativos)} de {train_mask.sum()}; minimo: {negativos['receiving_yards'].min():.0f}")
print(negativos[["player_display_name", "season", "week", "receptions", "receiving_yards"]].head(5).to_string())

filas de entrenamiento con receiving_yards < 0: 67 de 13795; minimo: -9
     player_display_name  season  week  receptions  receiving_yards
371          Eddie Royal    2016    12           1               -6
1199    Demaryius Thomas    2019     2           1               -1
1591        Randall Cobb    2019    15           1               -3
1899     Dontrelle Inman    2020     5           1               -2
2223     Jermaine Kearse    2018     2           1               -1


**Lectura.** Hay 67 filas de entrenamiento con yardas negativas, con un mínimo de −9. No son errores
de datos: son recepciones donde el receptor fue derribado detrás de la línea de golpeo, como se vio
en 2.1. `np.log1p()` no sirve para valores de −1 o menores, así que se usa un logaritmo con signo, `sign(x) · log1p(|x|)`, que acepta
negativos y deja el 0 en 0.

In [6]:
target = "receiving_yards"

def log_con_signo(x):
    return np.sign(x) * np.log1p(np.abs(x))

def inversa_log_con_signo(z):
    return np.sign(z) * np.expm1(np.abs(z))

imp_num_yds, imp_cat_yds = SimpleImputer(strategy="median"), SimpleImputer(strategy="most_frequent")
datos_train_yds = datos_bambi(train_mask, imp_num_yds, imp_cat_yds, ajustar=True)
datos_train_yds[f"{target}_log"] = log_con_signo(wr.loc[train_mask, target].to_numpy())

formula_yds = f"{target}_log ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo_yds = bmb.Model(formula_yds, datos_train_yds, family="gaussian")
t0 = time.time()
idata_yds = modelo_yds.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste: {time.time() - t0:.0f}s")

resumen_yds = az.summary(idata_yds)
print(f"r_hat maximo: {resumen_yds['r_hat'].max():.4f} | parametros con r_hat > 1.01: "
      f"{(resumen_yds['r_hat'] > 1.01).sum()} de {len(resumen_yds)}")

Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, receptions_career_avg, depth_team, receptions_season_avg, receptions_last3_avg, fantasy_points_ppr_last5_avg, total_implicito_equipo, receiving_yards_after_catch_career_avg, receiving_yards_career_avg, wopr_last3_avg, receiving_yards_last5_avg, fantasy_points_ppr_career_avg, receptions_last5_avg, fantasy_points_ppr_last3_avg, wopr_season_avg, target_share_x_ofensiva_equipo, targets_season_avg, draft_pick, receiving_first_downs_career_avg, receiving_yards_after_catch_season_avg, targets_last5_avg, edad, anios_experiencia, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 37 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste: 46s


r_hat maximo: 1.0182 | parametros con r_hat > 1.01: 4 de 534


### Volver a la escala de yardas sin perder la media

El modelo predice el promedio del logaritmo. Deshacer el logaritmo sobre ese promedio no da el
promedio de las yardas: da algo cercano a la mediana, que en una distribución con cola a la derecha
queda por debajo de la media. Como el proyecto necesita valores esperados, se compara también la
corrección de Duan (1983, *smearing*): sumar a cada predicción los residuos de entrenamiento en
escala logarítmica, deshacer el logaritmo y promediar. Se usa una muestra de 2,000 residuos.

In [7]:
pred_log_train = predecir_mu(modelo_yds, idata_yds, datos_train_yds.drop(columns=f"{target}_log"))
residuos = datos_train_yds[f"{target}_log"].to_numpy() - pred_log_train
muestra_residuos = np.random.default_rng(0).choice(residuos, size=2000, replace=False)

def retransformar(pred_log, corregir):
    if not corregir:
        return np.clip(inversa_log_con_signo(pred_log), 0, None)
    bloques = [inversa_log_con_signo(b[:, None] + muestra_residuos[None, :]).mean(axis=1)
               for b in np.array_split(pred_log, max(1, len(pred_log) // 1000))]
    return np.clip(np.concatenate(bloques), 0, None)

pred_sin_correccion = pd.Series(np.nan, index=wr.index)
pred_con_correccion = pd.Series(np.nan, index=wr.index)
for mascara in (val_mask, test_mask):
    pred_log = predecir_mu(modelo_yds, idata_yds, datos_bambi(mascara, imp_num_yds, imp_cat_yds))
    pred_sin_correccion[mascara] = retransformar(pred_log, corregir=False)
    pred_con_correccion[mascara] = retransformar(pred_log, corregir=True)

comparacion_yds = comparar("receiving_yards", {
    "XGBoost elegido": xgb_elegido("receiving_yards"),
    "Jerárquico log, sin corrección": pred_sin_correccion,
    "Jerárquico log, corrección de Duan": pred_con_correccion,
})
comparacion_yds.T

/tmp/ipykernel_7/2073560033.py:40: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


/tmp/ipykernel_7/2073560033.py:40: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


/tmp/ipykernel_7/2073560033.py:40: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  p = modelo.predict(idata, data=datos, kind="response_params", inplace=False, sample_new_groups=True)


,XGBoost elegido,"Jerárquico log, sin corrección","Jerárquico log, corrección de Duan"
val_rmse,29.50024,33.021956,41.652298
val_r2_oos,0.36552,0.20499,-0.264867
val_sesgo,1.472209,-11.072101,12.158252
val_pendiente_calibracion,1.066727,0.88482,0.44686
val_mae,21.985679,21.939743,28.630599
test_rmse,27.934865,32.183174,37.637348
test_r2_oos,0.374428,0.169687,-0.135592
test_sesgo,1.225055,-12.564668,7.012853
test_pendiente_calibracion,1.015915,0.912789,0.460723
test_mae,20.552169,21.148057,25.153348


**Lectura.** Ninguna de las dos versiones compite con el XGBoost elegido (RMSE de validación 29.50):

- **Sin corrección** (33.02, diferencia de +3.52 con intervalo de +3.00 a +4.07): subestima en
  promedio 11.1 yardas por receptor en validación y 12.6 en prueba. Es el sesgo de retransformación
  descrito arriba: deshacer el logaritmo del promedio da algo cercano a la mediana. De paso muestra
  otra vez por qué no se elige con MAE: esta versión tiene el MAE de validación más bajo de la tabla
  (21.94 contra 21.99), justamente por predecir cerca de la mediana.
- **Con la corrección de Duan** (41.65): se va al extremo contrario, con +12.2 yardas de sesgo y una
  pendiente de calibración de 0.45. La corrección supone que la dispersión de los residuos en escala
  logarítmica es la misma para todos los receptores. Si no lo es, multiplicar todas las predicciones
  por el mismo factor exagera las altas; la pendiente de 0.45 (predicciones demasiado extremas) es
  consistente con eso. El caso está registrado en [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md).

Una alternativa sería modelar la media directamente (por ejemplo, una familia para valores positivos
con enlace logarítmico y una parte aparte para los ceros y los negativos) en lugar de transformar el
resultado; no se prueba aquí.

## Cierre

El modelo jerárquico **no se adopta** para recepciones ni para yardas:

| Resultado | Jerárquico | RMSE de validación (jerárquico contra XGBoost elegido) | Diferencia (IC 95%) |
|---|---|---|---|
| `receptions` | Binomial Negativa + efecto por jugador | 2.039 contra 1.931 | +0.107 (+0.085 a +0.132) |
| `receiving_yards` | Gaussiana sobre log con signo, sin corrección | 33.02 contra 29.50 | +3.52 (+3.00 a +4.07) |
| `receiving_yards` | la misma, con corrección de Duan | 41.65 contra 29.50 | +12.15 (+10.99 a +13.42) |

Con [`4.7_jerarquico_touchdowns.ipynb`](4.7_jerarquico_touchdowns.ipynb), el resultado es el mismo en
los tres: el modelo jerárquico, lineal en sus 24 variables más un efecto por jugador, queda por
detrás del XGBoost con sus 25. En yardas aparece además un problema propio de transformar el
resultado: la escala logarítmica predice cerca de la mediana, y la corrección estándar no lo
resuelve. No se guardan estos modelos: no se adoptan y se regeneran corriendo este notebook.

Anterior: [`4.7_jerarquico_touchdowns.ipynb`](4.7_jerarquico_touchdowns.ipynb) · Siguiente:
[`4.9_ventana_entrenamiento.ipynb`](4.9_ventana_entrenamiento.ipynb)